In [ ]:
from dotenv import load_dotenv
import textwrap

from langchain import hub
from langchain_teddynote import logging
from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_upstage import UpstageEmbeddings
from langchain_openai import ChatOpenAI
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.runnables import chain
from langchain_teddynote.messages import stream_response
from langchain_teddynote.callbacks import StreamingCallback
from langchain_core.output_parsers import StrOutputParser, SimpleJsonOutputParser

from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

import warnings  # 경고 제거

In [ ]:
load_dotenv()

In [ ]:
logging.langsmith("langchain-summary-cmr")

# Clustering Map Refine

In [ ]:
FILE_PATH = "C:/Users/grego/experiment/AI4CEO/data/SPRI_AI_Brief_2023년12월호.pdf"

loader_cmr = PyPDFLoader(FILE_PATH)
docs_cmr = loader_cmr.load()
docs_cmr = docs_cmr[3:8]

print(f"총 페이지수: {len(docs_cmr)}")

In [ ]:
# 하나의 text로 모든 문서를 연결 -> 합쳐진 문자수는 약 28K
texts = "\n\n".join([doc.page_content for doc in docs])
len(texts)

text splitter

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

In [ ]:
split_docs = text_splitter.split_text(texts)

In [ ]:
len(split_docs)

Embedding

In [ ]:
embeddings_upstage = UpstageEmbeddings(model="solar-embedding-1-large-passage")
embeddings_openai = OpenAIEmbeddings()

In [ ]:
vectors_upstage = embeddings_upstage.embed_documents(split_docs)
vectors_openai = embeddings_openai.embed_documents(split_docs)

클러스터

In [ ]:
# KMeans로 79개 문서를 10개 클러스터로 나누기
num_clusters = 10

kmeans = KMeans(n_clusters=num_clusters, random_state=123).fit(vectors_upstage)

라벨링된 결과

In [ ]:
kmeans.labels_

In [ ]:
warnings.filterwarnings("ignore")

클러스터링 결과 시각화

In [ ]:
# t-SNE 수행 및 2차원으로 축소
tsne = TSNE(n_components=2, random_state=42)
reduced_data_tsne = tsne.fit_transform(np.array(vectors))

# seaborn 스타일 설정
sns.set_style("white")

# 축소된 데이터 플롯
plt.figure(figsize=(10, 8))
sns.scatterplot(
    x=reduced_data_tsne[:, 0],
    y=reduced_data_tsne[:, 1],
    hue=kmeans.labels_,
    palette="deep",
    s=100,
)
plt.xlabel("Dimension 1", fontsize=12)
plt.ylabel("Dimension 2", fontsize=12)
plt.title("Clustered Embeddings", fontsize=16)
plt.legend(title="Cluster", title_fontsize=12)

# 배경색 설정
plt.gcf().patch.set_facecolor("white")

plt.tight_layout()
plt.show()

각 클러스터의 중심점에 가장 가까운 임베딩 찾아서 저장하기

In [ ]:
# 가장 가까운 점들을 저장할 빈 리스트 생성
closest_indices = []

# 클러스터 수만큼 반복
for i in range(num_clusters):

    # 해당 클러스터 중심으로부터의 거리 목록 구하기
    distances = np.linalg.norm(vectors - kmeans.cluster_centers_[i], axis=1)

    # 가장 가까운 점의 인덱스 찾기 (argmin을 사용하여 최소 거리 찾기)
    closest_index = np.argmin(distances)

    # 해당 인덱스를 가장 가까운 인덱스 리스트에 추가
    closest_indices.append(closest_index)

In [ ]:
closest_indices

문서 요약

In [ ]:
# 순서대로 요약하기 위해 오름차순 정렬
selected_indices = sorted(closest_indices)
selected_indices

In [ ]:
# 선택된 10개 문서 출력
selected_docs = [Document(page_content=split_docs[doc]) for doc in selected_indices]
selected_docs

In [ ]:
# 요약 생성
refined_summary = map_refine_chain.invoke(selected_docs)

print(refined_summary)